In [1]:
# -*- coding: utf-8 -*-
"""
US required return batch runner (SAFE VERSION)
- SPY price는 배치 시작 시 1회만 확보/갱신 (티커마다 재호출 금지)
- RF(미국 국채금리)도 배치 시작 시 1회만 다운로드해서 전 티커 재사용
- 저장 지표를 "최소(권장)" / "전체" 중 선택 가능 (DB 폭발 방지)
- 체크포인트(처리 완료 티커 로그) + 실패 티커 재시도 로직
- MySQL PK(date,ticker,indicator) 기반 ON DUPLICATE KEY UPDATE
- NaN/inf 절대 저장 금지

※ 보안: DB 비밀번호/API KEY는 코드에 하드코딩하지 말고 환경변수 사용 권장
"""

import os
import time
import math
import json
import requests
import numpy as np
import pandas as pd
import pymysql
import FinanceDataReader as fdr
from datetime import datetime
from typing import Optional, Dict, Any, List, Tuple

# =========================================================
# 0) 설정
# =========================================================
DB_NAME = "investar"
TABLE_RESULT = "us_required_return_result"
DEFAULT_PORT = 3307

MARKET_TICKER = "SPY"

MAX_RETRY = 5
SLEEP_BETWEEN_CALLS = 0.35   # 기본 0.25는 전종목에서 429가 잦을 수 있어 상향(필요시 더 키우세요)

BETA_WINDOWS = [252, 750, 1250]  # beta_252, beta_750, beta_1250

# 저장 지표 모드
# - "minimal": DB 폭발 방지(권장) -> price_stock, beta_*, Re_* 만 저장
# - "full":   기존처럼 price/return/rf/E_Rm/Re/beta 전부 저장(테이블이 매우 커짐)
STORE_MODE = "minimal"  # "minimal" 또는 "full"

# 체크포인트/로그
CHECKPOINT_DIR = "_batch_checkpoint"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
DEFAULT_DONE_PATH = os.path.join(CHECKPOINT_DIR, "done_tickers.txt")
DEFAULT_FAIL_PATH = os.path.join(CHECKPOINT_DIR, "failed_tickers.txt")

# =========================================================
# 1) DB 연결 / 조회
# =========================================================
def get_conn(db_info: Dict[str, Any]):
    return pymysql.connect(
        host=db_info["host"],
        port=db_info.get("port", DEFAULT_PORT),
        user=db_info["user"],
        password=db_info["password"],
        db=db_info.get("database", DB_NAME),
        charset="utf8mb4",
        autocommit=False,
        cursorclass=pymysql.cursors.DictCursor
    )

def ensure_table_pk_hint():
    """
    테이블이 아래와 같은 PK 또는 UNIQUE INDEX를 반드시 가져야 합니다.
    PRIMARY KEY (date, ticker, indicator)
    """
    pass

def get_first_date_in_db(db_info: Dict[str, Any], ticker: str, indicator: str) -> Optional[pd.Timestamp]:
    sql = f"""
    SELECT MIN(date) AS first_date
    FROM {TABLE_RESULT}
    WHERE ticker=%s AND indicator=%s;
    """
    conn = get_conn(db_info)
    try:
        with conn.cursor() as cur:
            cur.execute(sql, (ticker, indicator))
            row = cur.fetchone()
            first_dt = row["first_date"] if row else None
    finally:
        conn.close()
    return pd.to_datetime(first_dt) if first_dt is not None else None

def get_last_date_in_db(db_info: Dict[str, Any], ticker: str, indicator: str) -> Optional[pd.Timestamp]:
    sql = f"""
    SELECT MAX(date) AS last_date
    FROM {TABLE_RESULT}
    WHERE ticker=%s AND indicator=%s;
    """
    conn = get_conn(db_info)
    try:
        with conn.cursor() as cur:
            cur.execute(sql, (ticker, indicator))
            row = cur.fetchone()
            last_dt = row["last_date"] if row else None
    finally:
        conn.close()
    return pd.to_datetime(last_dt) if last_dt is not None else None

def read_indicator_series(db_info: Dict[str, Any], ticker: str, indicator: str) -> pd.DataFrame:
    """
    returns: columns [date, value]
    """
    sql = f"""
    SELECT date, value
    FROM {TABLE_RESULT}
    WHERE ticker=%s AND indicator=%s
    ORDER BY date;
    """
    conn = get_conn(db_info)
    try:
        df = pd.read_sql(sql, conn, params=[ticker, indicator])
    finally:
        conn.close()

    if df.empty:
        return pd.DataFrame(columns=["date", "value"])

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.drop_duplicates(subset=["date"]).sort_values("date")
    return df

def fetch_price_stock_from_db_pivot(
    db_info: Dict[str, Any],
    ticker: str,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    indicator: str = "price_stock",
    table_name: str = TABLE_RESULT,
) -> pd.DataFrame:
    """
    Pivot 방식(MAX(CASE WHEN...))으로 price_stock만 추출
    return: [date, ticker, price_stock]
    """
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info.get("port", DEFAULT_PORT),
        user=db_info["user"],
        password=db_info["password"],
        database=db_info.get("database", DB_NAME),
        charset="utf8mb4",
    )

    try:
        where_clauses = ["ticker = %s"]
        params = [ticker]

        if start_date is not None:
            where_clauses.append("date >= %s")
            params.append(start_date)
        if end_date is not None:
            where_clauses.append("date <= %s")
            params.append(end_date)

        where_sql = "WHERE " + " AND ".join(where_clauses)

        query = f"""
            SELECT
                date,
                ticker,
                MAX(CASE WHEN indicator = %s THEN value END) AS `{indicator}`
            FROM {table_name}
            {where_sql}
            GROUP BY date, ticker
            ORDER BY date, ticker;
        """
        all_params = [indicator] + params
        df = pd.read_sql(query, conn, params=all_params)

        df["date"] = pd.to_datetime(df["date"].astype(str).str.strip(), errors="coerce")
        df[indicator] = pd.to_numeric(df[indicator], errors="coerce")
        df = df.dropna(subset=["date"])
        df = df.sort_values(["date", "ticker"]).reset_index(drop=True)
        return df
    finally:
        conn.close()


# =========================================================
# 2) Ticker universe (FDR listing) + 필터
# =========================================================
def get_filtered_us_tickers() -> List[str]:
    """
    NASDAQ / NYSE / AMEX 전체 상장사 정보를 기반으로
    특정 IndustryCode 앞 4자리에 해당하는 기업들을 제외하고
    최종 ticker 리스트를 반환.
    """
    nasdaq = fdr.StockListing('NASDAQ')
    nyse = fdr.StockListing('NYSE')
    amex = fdr.StockListing('AMEX')
    info_df = pd.concat([nasdaq, nyse, amex], ignore_index=True)

    exclude_prefixes = ["5510", "5730", "5530", "6010", "5910", "5120"]

    if "IndustryCode" not in info_df.columns or "Symbol" not in info_df.columns:
        # FDR 스키마가 바뀔 가능성 방어
        tickers = info_df.iloc[:, 0].dropna().astype(str).unique().tolist()
        tickers = [t.strip() for t in tickers if t.strip()]
        return tickers

    info_df["IndustryCode"] = info_df["IndustryCode"].astype(str)
    info_df["IndustryPrefix"] = info_df["IndustryCode"].str[:4]

    mask_exclude = info_df["IndustryPrefix"].isin(exclude_prefixes)
    excluded_df = info_df[mask_exclude]
    filtered_df = info_df[~mask_exclude].copy()

    tickers = filtered_df["Symbol"].dropna().astype(str).unique().tolist()
    tickers = [t.strip() for t in tickers if t.strip()]

    print(f"[INFO] 제외된 기업 수: {len(excluded_df)}")
    print(f"[INFO] 남은 기업 수: {len(filtered_df)}")
    print(f"[INFO] 티커 수: {len(tickers)}")
    return tickers


# =========================================================
# 3) FMP 호출 (가격)
# =========================================================
def _get_json(url: str, params: Dict[str, Any]) -> Any:
    last_err = None
    for k in range(MAX_RETRY):
        try:
            r = requests.get(url, params=params, timeout=30)
            if r.status_code == 429:
                time.sleep(1.0 + 0.7 * k)  # backoff 강화
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            time.sleep(0.7 + 0.7 * k)
    raise RuntimeError(f"FMP request failed after retries. last_err={last_err}")

def fetch_fmp_price(
    symbol: str,
    api_key: str,
    start_date: str,
    min_start_date: str = "2015-01-01"
) -> pd.DataFrame:
    """
    return columns [date, price] (adjClose 우선)
    start_date가 min_start_date보다 최근이면 min_start_date로 강제 당김
    """
    if pd.to_datetime(start_date) > pd.to_datetime(min_start_date):
        start_date = min_start_date

    url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{symbol}"
    params = {"from": start_date, "apikey": api_key}

    js = _get_json(url, params=params)
    hist = js.get("historical", []) if isinstance(js, dict) else []
    if not hist:
        return pd.DataFrame(columns=["date", "price"])

    df = pd.DataFrame(hist)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    price_col = "adjClose" if "adjClose" in df.columns else "close"
    df[price_col] = pd.to_numeric(df[price_col], errors="coerce")
    df = df[["date", price_col]].rename(columns={price_col: "price"})
    df = df.dropna(subset=["price"]).drop_duplicates("date").sort_values("date")
    return df


# =========================================================
# 4) RF(국채 금리) - 배치 1회 다운로드 후 재사용
# =========================================================
def fetch_us_treasury_yields(start_date: str, end_date: Optional[str] = None) -> pd.DataFrame:
    """
    old_version 방식 유지:
    1) FDR(FRED:DGS1,3,5) 시도
    2) 실패 시 Yahoo (^IRX, ^FVX) fallback + 3Y는 선형보간
    return columns: [rf_1y, rf_3y, rf_5y] in decimals
    """
    if end_date is None:
        end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

    # 1) FRED via FDR
    try:
        y1 = fdr.DataReader("FRED:DGS1", start_date, end_date)
        y3 = fdr.DataReader("FRED:DGS3", start_date, end_date)
        y5 = fdr.DataReader("FRED:DGS5", start_date, end_date)

        idx = y1.index.union(y3.index).union(y5.index)
        out = pd.DataFrame(index=idx).sort_index()
        out["rf_1y"] = pd.to_numeric(y1.iloc[:, 0], errors="coerce") / 100.0
        out["rf_3y"] = pd.to_numeric(y3.iloc[:, 0], errors="coerce") / 100.0
        out["rf_5y"] = pd.to_numeric(y5.iloc[:, 0], errors="coerce") / 100.0
        return out
    except Exception:
        pass

    # 2) Yahoo fallback
    y1 = fdr.DataReader("^IRX", start_date, end_date)  # 13-week proxy
    y5 = fdr.DataReader("^FVX", start_date, end_date)  # 5Y

    idx = y1.index.union(y5.index)
    out = pd.DataFrame(index=idx).sort_index()
    out["rf_1y"] = pd.to_numeric(y1["Close"], errors="coerce") / 100.0
    out["rf_5y"] = pd.to_numeric(y5["Close"], errors="coerce") / 100.0
    out["rf_3y"] = out["rf_1y"] + (out["rf_5y"] - out["rf_1y"]) * (3 - 1) / (5 - 1)
    return out


# =========================================================
# 5) 계산 (old_version 유지)
# =========================================================
def rolling_beta(ret_stock: pd.Series, ret_mkt: pd.Series, window: int) -> pd.Series:
    cov = ret_stock.rolling(window).cov(ret_mkt)
    var = ret_mkt.rolling(window).var()
    return cov / var

def build_features(price_stock_df: pd.DataFrame, price_mkt_df: pd.DataFrame, rf_df: pd.DataFrame) -> pd.DataFrame:
    """
    price_stock_df: [date, price_stock]
    price_mkt_df:   [date, price_mkt]
    rf_df index=date with [rf_1y, rf_3y, rf_5y]
    """
    df = pd.merge(price_stock_df, price_mkt_df, on="date", how="inner")
    df = df.sort_values("date").drop_duplicates("date")

    df["price_stock"] = pd.to_numeric(df["price_stock"], errors="coerce")
    df["price_mkt"]   = pd.to_numeric(df["price_mkt"], errors="coerce")
    df = df.dropna(subset=["price_stock", "price_mkt"])

    df["ret_stock"] = df["price_stock"].pct_change()
    df["ret_mkt"]   = df["price_mkt"].pct_change()

    # beta_252/750/1250
    for w in BETA_WINDOWS:
        df[f"beta_{w}"] = rolling_beta(df["ret_stock"], df["ret_mkt"], w)

    # rf merge (ffill)
    if rf_df is None or rf_df.empty:
        df["rf_1y"] = np.nan
        df["rf_3y"] = np.nan
        df["rf_5y"] = np.nan
    else:
        rf2 = rf_df.copy().sort_index()
        rf2 = rf2.reindex(pd.to_datetime(df["date"])).ffill()
        df["rf_1y"] = rf2["rf_1y"].values
        df["rf_3y"] = rf2["rf_3y"].values
        df["rf_5y"] = rf2["rf_5y"].values

    # E_Rm (old_version: mean * 252)
    df["E_Rm_1y"] = df["ret_mkt"].rolling(252).mean() * 252
    df["E_Rm_3y"] = df["ret_mkt"].rolling(750).mean() * 252
    df["E_Rm_5y"] = df["ret_mkt"].rolling(1250).mean() * 252

    # Required return (각 horizon에 맞는 beta 사용: old_version 유지)
    df["Re_1y"] = df["rf_1y"] + df["beta_252"]   * (df["E_Rm_1y"] - df["rf_1y"])
    df["Re_3y"] = df["rf_3y"] + df["beta_750"]   * (df["E_Rm_3y"] - df["rf_3y"])
    df["Re_5y"] = df["rf_5y"] + df["beta_1250"]  * (df["E_Rm_5y"] - df["rf_5y"])

    return df


# =========================================================
# 6) MySQL 저장: NaN/inf 금지 + updated_at 없음 + PK 중복 방지
# =========================================================
def _to_mysql_float(x: Any) -> Optional[float]:
    if x is None:
        return None
    try:
        v = float(x)
    except Exception:
        return None
    if math.isnan(v) or math.isinf(v):
        return None
    return v

def sanitize_long_for_mysql(df: pd.DataFrame) -> pd.DataFrame:
    """
    df columns = [date, ticker, indicator, value]
    """
    out = df.copy()

    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out = out[out["date"].notna()]
    out["date"] = out["date"].dt.date

    out["ticker"] = out["ticker"].astype(str)
    out["indicator"] = out["indicator"].astype(str)

    out = out[(out["ticker"].str.lower() != "nan") & (out["ticker"].str.lower() != "none")]
    out = out[(out["indicator"].str.lower() != "nan") & (out["indicator"].str.lower() != "none")]

    out["value"] = pd.to_numeric(out["value"], errors="coerce")
    out["value"] = out["value"].apply(_to_mysql_float)

    # 최종 중복 제거
    out = out.drop_duplicates(subset=["date", "ticker", "indicator"])
    return out

def upsert_long_df(
    db_info: Dict[str, Any],
    long_df: pd.DataFrame,
    batch_size_rows: int = 50_000,
    batch_size_ticker: int = 50,
    drop_null_values: bool = True
) -> None:
    """
    PK(date,ticker,indicator) 기반으로 중복 방지.
    """
    if long_df is None or long_df.empty:
        return

    df = sanitize_long_for_mysql(long_df)
    if drop_null_values:
        df = df.dropna(subset=["value"])
    if df.empty:
        return

    tickers = sorted(df["ticker"].unique())
    n = len(tickers)
    print(f"[INFO] upsert 대상 ticker={n}, rows={len(df):,}")

    insert_sql = f"""
    INSERT INTO {TABLE_RESULT} (date, ticker, indicator, value)
    VALUES (%s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        value = VALUES(value);
    """

    for i in range(0, n, batch_size_ticker):
        batch_tickers = tickers[i:i+batch_size_ticker]
        batch = df[df["ticker"].isin(batch_tickers)].copy()
        batch = batch.sort_values(["ticker", "date", "indicator"])
        rows = list(batch[["date", "ticker", "indicator", "value"]].itertuples(index=False, name=None))

        conn = get_conn(db_info)
        try:
            with conn.cursor() as cur:
                for j in range(0, len(rows), batch_size_rows):
                    chunk = rows[j:j+batch_size_rows]

                    # 샘플 검증
                    for _r in chunk[:10]:
                        vv = _r[3]
                        if isinstance(vv, float) and (math.isnan(vv) or math.isinf(vv)):
                            raise ValueError(f"Still has NaN/inf in chunk sample: {_r}")

                    cur.executemany(insert_sql, chunk)
                    conn.commit()
            print(f"[OK] batch {i//batch_size_ticker+1}: tickers={len(batch_tickers)}, rows={len(rows):,}")
        except Exception:
            conn.rollback()
            raise
        finally:
            conn.close()


# =========================================================
# 7) 가격 확보: DB price_stock → 부족분만 FMP → DB 저장
#    (SPY 포함. 단, SPY는 배치에서 1회만 호출하도록 run_batch에서 제어)
# =========================================================
def ensure_price_series(
    db_info: Dict[str, Any],
    ticker: str,
    api_key: str,
    indicator_name: str = "price_stock",
    today_iso: Optional[str] = None,
    min_start_date: str = "2015-01-01",
) -> pd.DataFrame:
    """
    - DB에 없으면 min_start_date부터 full fetch 후 저장
    - DB 시작이 min_start_date보다 늦으면 과거 backfill 저장
    - DB 마지막이 today보다 뒤처지면 forward 구간 추가 fetch 후 저장
    return: [date, indicator_name]
    """
    if today_iso is None:
        today_iso = datetime.utcnow().date().isoformat()

    df_db = read_indicator_series(db_info, ticker, indicator_name)  # [date,value]
    df_db = df_db.rename(columns={"value": indicator_name})

    first_dt = get_first_date_in_db(db_info, ticker, indicator_name)
    last_dt  = get_last_date_in_db(db_info, ticker, indicator_name)

    # (A) DB에 아무것도 없으면: min_start_date부터 채움
    if last_dt is None:
        df_new = fetch_fmp_price(ticker, api_key, start_date=min_start_date, min_start_date=min_start_date)\
                    .rename(columns={"price": indicator_name})
        time.sleep(SLEEP_BETWEEN_CALLS)

        if not df_new.empty:
            long_new = df_new.assign(ticker=ticker).melt(
                id_vars=["date", "ticker"], value_vars=[indicator_name],
                var_name="indicator", value_name="value"
            )
            upsert_long_df(db_info, long_new, drop_null_values=True)

        return df_new.drop_duplicates("date").sort_values("date")

    # (B) 과거 backfill (DB 시작이 min_start_date보다 늦으면)
    df_back = pd.DataFrame(columns=["date", indicator_name])
    if first_dt is not None and first_dt.date().isoformat() > min_start_date:
        back_end = (first_dt - pd.Timedelta(days=1)).date().isoformat()
        tmp = fetch_fmp_price(ticker, api_key, start_date=min_start_date, min_start_date=min_start_date)\
                .rename(columns={"price": indicator_name})
        df_back = tmp[tmp["date"] <= pd.to_datetime(back_end)]
        time.sleep(SLEEP_BETWEEN_CALLS)

        if not df_back.empty:
            long_back = df_back.assign(ticker=ticker).melt(
                id_vars=["date", "ticker"], value_vars=[indicator_name],
                var_name="indicator", value_name="value"
            )
            upsert_long_df(db_info, long_back, drop_null_values=True)

    # (C) 최신 forward-fill (last_dt+1 이후)
    start_missing = (last_dt + pd.Timedelta(days=1)).date().isoformat()
    df_fwd = pd.DataFrame(columns=["date", indicator_name])
    if start_missing <= today_iso:
        df_fwd = fetch_fmp_price(ticker, api_key, start_date=start_missing, min_start_date=min_start_date)\
                    .rename(columns={"price": indicator_name})
        time.sleep(SLEEP_BETWEEN_CALLS)

        if not df_fwd.empty:
            long_fwd = df_fwd.assign(ticker=ticker).melt(
                id_vars=["date", "ticker"], value_vars=[indicator_name],
                var_name="indicator", value_name="value"
            )
            upsert_long_df(db_info, long_fwd, drop_null_values=True)

    # (D) 최종 결합 리턴
    df_full = pd.concat([df_db, df_back, df_fwd], ignore_index=True)
    df_full["date"] = pd.to_datetime(df_full["date"], errors="coerce")
    df_full = df_full.dropna(subset=["date"]).drop_duplicates("date").sort_values("date")
    return df_full


# =========================================================
# 8) 티커 1개 업데이트 (SPY/RF는 외부에서 주입받아 재사용)
# =========================================================
def update_one_ticker(
    db_info: Dict[str, Any],
    ticker: str,
    api_key: str,
    spy_price_df: pd.DataFrame,   # 반드시 [date, price_mkt] 형태
    rf_df: pd.DataFrame,          # index=date, columns rf_1y, rf_3y, rf_5y
    today_iso: Optional[str] = None,
    min_start_date: str = "2015-01-01",
    store_mode: str = "minimal",  # "minimal" 또는 "full"
) -> Tuple[bool, str]:
    """
    return: (success, message)
    """
    if today_iso is None:
        today_iso = datetime.utcnow().date().isoformat()

    # 1) 종목 가격 확보(필요분만 FMP) + DB 저장
    px_stock = ensure_price_series(
        db_info=db_info,
        ticker=ticker,
        api_key=api_key,
        indicator_name="price_stock",
        today_iso=today_iso,
        min_start_date=min_start_date
    )
    if px_stock.empty:
        return (False, f"{ticker}: price_stock empty")

    # 2) SPY 가격 DF는 배치에서 확보된 것을 사용
    if spy_price_df is None or spy_price_df.empty:
        return (False, "SPY price df empty")

    price_stock_df = px_stock[["date", "price_stock"]].copy()
    price_mkt_df   = spy_price_df[["date", "price_mkt"]].copy()

    # 3) 계산
    feat = build_features(price_stock_df, price_mkt_df, rf_df)
    if feat.empty:
        return (False, f"{ticker}: feature df empty after merge")

    feat["ticker"] = ticker

    # 4) 저장 컬럼 선택
    if store_mode == "full":
        keep_cols = [
            "price_stock","price_mkt","ret_stock","ret_mkt",
            "beta_252","beta_750","beta_1250",
            "rf_1y","rf_3y","rf_5y",
            "E_Rm_1y","E_Rm_3y","E_Rm_5y",
            "Re_1y","Re_3y","Re_5y",
        ]
    else:
        # minimal(권장): DB 크기 폭발 방지
        keep_cols = [
            "price_stock",
            "beta_252","beta_750","beta_1250",
            "Re_1y","Re_3y","Re_5y",
        ]

    keep_cols = [c for c in keep_cols if c in feat.columns]

    long_df = feat[["date","ticker"] + keep_cols].melt(
        id_vars=["date","ticker"],
        value_vars=keep_cols,
        var_name="indicator",
        value_name="value"
    )

    # 5) NaN/inf 제거(최종 방어는 upsert 내부에서도 함)
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    long_df = long_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["value"])

    if long_df.empty:
        return (False, f"{ticker}: all values NaN after sanitize")

    # 6) 저장
    upsert_long_df(db_info, long_df, drop_null_values=True)
    return (True, f"{ticker}: saved rows={len(long_df):,} ({store_mode})")


# =========================================================
# 9) 배치 실행 (체크포인트/실패 재시도 포함)
# =========================================================
def _load_set(path: str) -> set:
    if not os.path.exists(path):
        return set()
    with open(path, "r", encoding="utf-8") as f:
        return set([line.strip() for line in f if line.strip()])

def _append_line(path: str, line: str):
    with open(path, "a", encoding="utf-8") as f:
        f.write(line.strip() + "\n")

def prepare_spy_and_rf(
    db_info: Dict[str, Any],
    api_key: str,
    today_iso: str,
    min_start_date: str = "2015-01-01",
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    1) SPY price_stock 확보/갱신 1회
    2) RF는 SPY 기간 기준으로 1회 다운로드
    return:
      spy_price_df: [date, price_mkt]
      rf_df: index=date, columns rf_1y, rf_3y, rf_5y
    """
    print("[STEP] Ensure SPY price series (once)")
    spy_px = ensure_price_series(
        db_info=db_info,
        ticker=MARKET_TICKER,
        api_key=api_key,
        indicator_name="price_stock",
        today_iso=today_iso,
        min_start_date=min_start_date
    )

    if spy_px.empty:
        raise RuntimeError("SPY price series is empty; cannot proceed.")

    spy_price_df = spy_px.rename(columns={"price_stock": "price_mkt"})[["date", "price_mkt"]].copy()
    spy_price_df["date"] = pd.to_datetime(spy_price_df["date"])
    spy_price_df = spy_price_df.dropna(subset=["date","price_mkt"]).drop_duplicates("date").sort_values("date")

    start_for_rf = spy_price_df["date"].min().date().isoformat()
    end_for_rf   = spy_price_df["date"].max().date().isoformat()

    print(f"[STEP] Fetch RF once for range: {start_for_rf} ~ {end_for_rf}")
    rf_df = fetch_us_treasury_yields(start_for_rf, end_for_rf)
    if rf_df is None or rf_df.empty:
        print("[WARN] RF df empty; required return may be NaN for many rows.")

    return spy_price_df, rf_df

def run_batch(
    db_info: Dict[str, Any],
    tickers: List[str],
    api_key: str,
    start_idx: int = 0,
    min_start_date: str = "2015-01-01",
    store_mode: str = "minimal",
    done_path: str = DEFAULT_DONE_PATH,
    fail_path: str = DEFAULT_FAIL_PATH,
    skip_done: bool = True,
    retry_failed_once: bool = True,
) -> None:
    today_iso = datetime.utcnow().date().isoformat()

    done_set = _load_set(done_path) if skip_done else set()
    fail_set = set()

    # 1) SPY & RF 준비 (배치 1회만)
    spy_price_df, rf_df = prepare_spy_and_rf(
        db_info=db_info,
        api_key=api_key,
        today_iso=today_iso,
        min_start_date=min_start_date
    )

    # 2) 본 배치
    total = len(tickers)
    print(f"[RUN] total tickers={total}, start_idx={start_idx}, store_mode={store_mode}, skip_done={skip_done}")

    for idx, t in enumerate(tickers[start_idx:], start=start_idx):
        t = str(t).strip()
        if not t:
            continue
        if skip_done and (t in done_set):
            if (idx % 200) == 0:
                print(f"[SKIP] idx={idx} {t} (already done)")
            continue

        try:
            ok, msg = update_one_ticker(
                db_info=db_info,
                ticker=t,
                api_key=api_key,
                spy_price_df=spy_price_df,
                rf_df=rf_df,
                today_iso=today_iso,
                min_start_date=min_start_date,
                store_mode=store_mode
            )
            if ok:
                print(f"[OK] idx={idx}/{total-1} {msg}")
                _append_line(done_path, t)
                done_set.add(t)
            else:
                print(f"[FAIL] idx={idx}/{total-1} {msg}")
                _append_line(fail_path, t)
                fail_set.add(t)

        except Exception as e:
            print(f"[ERROR] idx={idx}/{total-1} {t}: {e}")
            _append_line(fail_path, t)
            fail_set.add(t)

    # 3) 실패 티커 1회 재시도
    if retry_failed_once and fail_set:
        print(f"[RETRY] failed tickers={len(fail_set)} (one more pass)")
        # 재시도 리스트는 fail_path에 이미 적혀 있으니 여기서는 메모리 fail_set 사용
        still_fail = set()
        for i, t in enumerate(sorted(fail_set)):
            try:
                ok, msg = update_one_ticker(
                    db_info=db_info,
                    ticker=t,
                    api_key=api_key,
                    spy_price_df=spy_price_df,
                    rf_df=rf_df,
                    today_iso=today_iso,
                    min_start_date=min_start_date,
                    store_mode=store_mode
                )
                if ok:
                    print(f"[RETRY-OK] {msg}")
                    _append_line(done_path, t)
                    done_set.add(t)
                else:
                    print(f"[RETRY-FAIL] {msg}")
                    still_fail.add(t)
            except Exception as e:
                print(f"[RETRY-ERROR] {t}: {e}")
                still_fail.add(t)

        print(f"[DONE] retry finished. still_fail={len(still_fail)}")
    else:
        print("[DONE] batch finished.")


# =========================================================
# 10) 실행 예시(main)
# =========================================================
# if __name__ == "__main__":
#     # ----------------------------
#     # (1) DB / API 설정 (환경변수 권장)
#     # ----------------------------
#     # 예:
#     # setx INVESTAR_DB_PW "A3!~"
#     # setx FMP_API_KEY "hT0g..."
#     DB_PASSWORD = os.getenv("INVESTAR_DB_PW", "A3!~")  # 가능하면 환경변수로!
#     API_KEY     = os.getenv("FMP_API_KEY", "hT0g")     # 가능하면 환경변수로!
#
#     # 사용하시는 get_db_host() 유지
#     from DATA.stock_invest_function import get_db_host
#
#     db_info = {
#         "host": get_db_host(),
#         "port": 3307,
#         "user": "st2",
#         "password": DB_PASSWORD,
#         "database": "investar",
#     }
#
#     # ----------------------------
#     # (2) 티커 리스트
#     # ----------------------------
#     tickers = get_filtered_us_tickers()
#
#     # 필요하면 구간 슬라이스(예: 1000개만 테스트)
#     # tickers = tickers[:1000]
#
#     # ----------------------------
#     # (3) 배치 실행
#     # ----------------------------
#     run_batch(
#         db_info=db_info,
#         tickers=tickers,
#         api_key=API_KEY,
#         start_idx=0,
#         min_start_date="2015-01-01",
#         store_mode=STORE_MODE,         # "minimal" 권장
#         done_path=DEFAULT_DONE_PATH,
#         fail_path=DEFAULT_FAIL_PATH,
#         skip_done=True,
#         retry_failed_once=True,
#     )


In [3]:
from DATA.stock_invest_function import get_db_host

# 1) DB 설정.햣
db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",   # 실제 비밀번호
    "database": "investar",
}

API_KEY = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'

# 1) SPY 가격을 2015부터 채우기(과거 backfill + 최신 갱신)
ensure_price_series(db_info, "SPY", API_KEY, min_start_date="2015-01-01")

# 2) NVDA 가격도 2015부터 채우기
# ensure_price_series(db_info, "NVDA",API_KEY, min_start_date="2015-01-01")
#
# # 3) NVDA required return 재계산 후 DB 저장
# update_one_ticker(db_info, "NVDA", API_KEY, store_prices_and_returns=True)

tickers  = get_filtered_us_tickers()
# # TICKER_LIST = TICKER_LIST[1000:2000]
# tickers = ["PLMR", "LLY", "AMD"]
# tickers  = ["AAPL", "MSFT", "NVDA"]  # 예시

run_batch(
    db_info=db_info,
    tickers=tickers,
    api_key=API_KEY,
    start_idx=0,
    min_start_date="2015-01-01",
    store_mode=STORE_MODE,         # "minimal" 권장
    done_path=DEFAULT_DONE_PATH,
    fail_path=DEFAULT_FAIL_PATH,
    skip_done=True,
    retry_failed_once=True,
)

[INFO] upsert 대상 ticker=1, rows=2,773
[OK] batch 1: tickers=1, rows=2,773
[STEP] Ensure SPY price series (once)
[INFO] upsert 대상 ticker=1, rows=2,773
[OK] batch 1: tickers=1, rows=2,773
[STEP] Fetch RF once for range: 2015-01-02 ~ 2026-01-12
[RUN] total tickers=3, start_idx=0, store_mode=minimal, skip_done=True
[INFO] upsert 대상 ticker=1, rows=1,694
[OK] batch 1: tickers=1, rows=1,694
[INFO] upsert 대상 ticker=1, rows=7,354
[OK] batch 1: tickers=1, rows=7,354
[OK] idx=0/2 PLMR: saved rows=7,354 (minimal)
[INFO] upsert 대상 ticker=1, rows=2,773
[OK] batch 1: tickers=1, rows=2,773
[INFO] upsert 대상 ticker=1, rows=14,907
[OK] batch 1: tickers=1, rows=14,907
[OK] idx=1/2 LLY: saved rows=14,907 (minimal)
[INFO] upsert 대상 ticker=1, rows=2,773
[OK] batch 1: tickers=1, rows=2,773
[INFO] upsert 대상 ticker=1, rows=14,907
[OK] batch 1: tickers=1, rows=14,907
[OK] idx=2/2 AMD: saved rows=14,907 (minimal)
[DONE] batch finished.


In [4]:
aapl_price = fetch_price_stock_from_db_pivot(db_info, ticker="AMD")
print(aapl_price.head())
print(aapl_price.tail())
print(len(aapl_price))

        date ticker  price_stock
0 2011-01-03    AMD          NaN
1 2011-01-04    AMD          NaN
2 2011-01-05    AMD          NaN
3 2011-01-06    AMD          NaN
4 2011-01-07    AMD          NaN
           date ticker  price_stock
3774 2026-01-06    AMD       214.35
3775 2026-01-07    AMD       210.02
3776 2026-01-08    AMD       204.68
3777 2026-01-09    AMD       203.17
3778 2026-01-12    AMD       207.69
3779


In [69]:
########## 데이터 입력 테스트 코드

In [5]:
def get_indicator_list_from_db(db_info: Dict,
                               table_name: str = "us_required_return_result"
                               ) -> List[str]:
    """
    큰 테이블 전체를 읽지 않고,
    DB에서 DISTINCT indicator 이름만 추출.
    """
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    query = f"SELECT DISTINCT indicator FROM {table_name};"

    try:
        df = pd.read_sql(query, conn)
    finally:
        conn.close()

    return sorted(df["indicator"].dropna().tolist())


def fetch_required_return_pivot_from_db(
    db_info: Dict,
    table_name: str = "us_required_return_result",
    indicators: Optional[List[str]] = None,
    start_date: Optional[str] = None,   # "YYYY-MM-DD"
    end_date: Optional[str] = None,     # "YYYY-MM-DD"
    tickers: Optional[List[str]] = None
) -> pd.DataFrame:
    """
    us_required_return_result 테이블에서 직접 pivot 형태로 SELECT.

    - index: date, ticker
    - columns: indicator 이름들
    - values: value (MAX(CASE WHEN ...) 사용)

    큰 테이블 전체를 안 읽고, SQL에서 집계해서 가져오기 때문에 메모리 부담이 훨씬 적다.
    """

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        # 1) indicator 리스트가 안 들어오면, DB에서 DISTINCT로 먼저 가져오기
        if indicators is None:
            indicators = get_indicator_list_from_db(db_info, table_name)

        # 2) indicator별 CASE WHEN 절 만들기
        case_clauses = []
        for ind in indicators:
            # 컬럼 alias에 공백/특수문자가 있으면 쿼리가 깨질 수 있으므로 간단히 감싸줌
            safe_alias = ind.replace("`", "")  # 백틱 제거
            clause = f"MAX(CASE WHEN indicator = %s THEN value END) AS `{safe_alias}`"
            case_clauses.append(clause)

        case_sql = ",\n       ".join(case_clauses)

        # 3) WHERE 절 동적으로 생성
        where_clauses = []
        params = []

        if start_date is not None:
            where_clauses.append("date >= %s")
            params.append(start_date)

        if end_date is not None:
            where_clauses.append("date <= %s")
            params.append(end_date)

        if tickers is not None and len(tickers) > 0:
            # IN (%s, %s, ...)
            tick_placeholders = ", ".join(["%s"] * len(tickers))
            where_clauses.append(f"ticker IN ({tick_placeholders})")
            params.extend(tickers)

        where_sql = ""
        if where_clauses:
            where_sql = "WHERE " + " AND ".join(where_clauses)

        # 4) 전체 쿼리 조립
        #    date, ticker별로 indicator를 열로 펼친 형태
        query = f"""
            SELECT
                date,
                ticker,
                {case_sql}
            FROM {table_name}
            {where_sql}
            GROUP BY date, ticker
            ORDER BY date, ticker;
        """

        # indicator 값들을 CASE WHEN의 %s에 넣어줌
        # CASE WHEN indicator = %s THEN value END  → 각 indicator마다 하나씩
        case_params = indicators[:]  # 각 CASE WHEN 한 번씩
        all_params = case_params + params

        df_pivot = pd.read_sql(query, conn, params=all_params)

        # date를 datetime으로 변환 (필요시)
        df_pivot["date"] = pd.to_datetime(df_pivot["date"])
        return df_pivot

    finally:
        conn.close()

In [8]:
company_ticker = 'AAPL'
item_name = 'roe'

tickers = ['AAPL']

re_df = fetch_required_return_pivot_from_db(
    db_info=db_info,
    table_name="us_required_return_result",
    indicators=['Re_5y'],        # ← 리스트로!
    start_date="2020-01-01",
    end_date=None,
    tickers=tickers
)

print(re_df.head())

        date ticker     Re_5y
0 2020-01-02   AAPL  0.152094
1 2020-01-03   AAPL  0.152494
2 2020-01-06   AAPL  0.150513
3 2020-01-07   AAPL  0.149214
4 2020-01-08   AAPL  0.149212


In [7]:
re_df.tail(12)

,date,ticker,Re_5y
1503,2025-12-24,AMD,0.274768
1504,2025-12-26,AMD,0.272175
1505,2025-12-29,AMD,0.268629
1506,2025-12-30,AMD,0.261777
1507,2025-12-31,AMD,0.256049
1508,2026-01-02,AMD,0.259633
1509,2026-01-05,AMD,0.262429
1510,2026-01-06,AMD,0.263547
1511,2026-01-07,AMD,0.263952
1512,2026-01-08,AMD,0.266382


In [9]:
re_df.tail()

,date,ticker,Re_5y
1509,2026-01-05,AAPL,0.174737
1510,2026-01-06,AAPL,0.175397
1511,2026-01-07,AAPL,0.175502
1512,2026-01-08,AAPL,0.177166
1513,2026-01-09,AAPL,0.176853


In [14]:
# # 1) DB 설정
# db_info = {
#     "host": get_db_host(),
#     "port": 3307,
#     "user": "stox7412",
#     "password": "Apt106503!~",   # 실제 비밀번호
#     "database": "investar",
# }
#
# API_KEY = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'
#
# # TICKER_LIST = ["AAPL", "MSFT", "NVDA"]  # 예시
#
# TICKER_LIST = get_filtered_us_tickers()
# # TICKER_LIST = TICKER_LIST[1000:2000]
# TICKER_LIST = ["BG", "PLMR", "LLY", "AMD", "DASH", "LULU", "CAT"]

100%|██████████| 299/299 [00:00<00:00, 1138.13it/s]

제외된 기업 수: 1809
남은 기업 수: 4994
티커 수: 4994


In [53]:
diag = fetch_required_return_pivot_from_db(
    db_info=db_info,
    table_name="us_required_return_result",
    indicators=["beta_1250","rf_5y","E_Rm_5y","Re_5y","price_stock","price_mkt"],
    start_date="2015-01-01",
    tickers=["NVDA"]
)
print(diag.tail(15)[["date","ticker","beta_1250","rf_5y","E_Rm_5y","Re_5y"]])
print(diag[["beta_1250","rf_5y","E_Rm_5y","Re_5y"]].isna().sum())

           date ticker  beta_1250 rf_5y E_Rm_5y  Re_5y
2757 2025-12-18   NVDA        NaN  None    None    NaN
2758 2025-12-19   NVDA        NaN  None    None    NaN
2759 2025-12-22   NVDA        NaN  None    None    NaN
2760 2025-12-23   NVDA        NaN  None    None    NaN
2761 2025-12-24   NVDA        NaN  None    None    NaN
2762 2025-12-26   NVDA        NaN  None    None    NaN
2763 2025-12-29   NVDA        NaN  None    None    NaN
2764 2025-12-30   NVDA        NaN  None    None    NaN
2765 2025-12-31   NVDA        NaN  None    None    NaN
2766 2026-01-02   NVDA        NaN  None    None    NaN
2767 2026-01-05   NVDA        NaN  None    None    NaN
2768 2026-01-06   NVDA        NaN  None    None    NaN
2769 2026-01-07   NVDA        NaN  None    None    NaN
2770 2026-01-08   NVDA        NaN  None    None    NaN
2771 2026-01-09   NVDA        NaN  None    None    NaN
beta_1250      29
rf_5y        2772
E_Rm_5y      2772
Re_5y          29
dtype: int64


In [56]:
def get_minmax_date_safe(db_info, ticker, indicator="price_stock", table="us_required_return_result"):
    conn = get_conn(db_info)
    try:
        sql = f"""
        SELECT
            MIN(STR_TO_DATE(date, '%%Y-%%m-%%d')) AS min_date,
            MAX(STR_TO_DATE(date, '%%Y-%%m-%%d')) AS max_date,
            COUNT(*) AS n
        FROM {table}
        WHERE ticker=%s AND indicator=%s
          AND STR_TO_DATE(date, '%%Y-%%m-%%d') IS NOT NULL;
        """
        df = pd.read_sql(sql, conn, params=[ticker, indicator])
    finally:
        conn.close()
    return df

In [57]:
print("SPY price_stock:", get_minmax_date_safe(db_info, "SPY", "price_stock"))
print("NVDA price_stock:", get_minmax_date_safe(db_info, "NVDA", "price_stock"))
print("NVDA price_mkt :", get_minmax_date_safe(db_info, "NVDA", "price_mkt"))

SPY price_stock:    min_date  max_date  n
0  min_date  max_date  n
NVDA price_stock:    min_date  max_date  n
0  min_date  max_date  n
NVDA price_mkt :    min_date  max_date  n
0  min_date  max_date  n
